# ChebAr, larger truncation — convergence and Ritz state

**Cat size.** `alpha_sq` is `|alpha|^2` (`epsilon_d = 2 alpha_sq g`, `docs/models_ats.md` §2). Saved outputs predate the 2026-09 fix and print twice the current values for the same physical point.

Companion to `notebooks/solvers/cheb_ar.ipynb`, at a bigger Fock truncation
(`n_a, n_b = 30, 11`) and with the Arnoldi *convergence history* plotted rather
than just the final rate.

Ported to the anb-compute Ray cluster: the pipeline is
`floquet_lindblad.pipeline.solve_point`, run as one GPU task, and everything below is
plotted locally from plain numpy.

## Running it

1. **Kernel**: the workspace pixi environment,
   `/workspace/.pixi/envs/default/bin/python` (**Python 3.13.x**, `ray 2.56.0`)
   — these must match the cluster.
2. **Vault session**, once (lasts about a week):

   ```bash
   export VAULT_ADDR=https://vault.int.alice-bob.com
   vault login -method=oidc role=research_core_user
   ```

3. Run all cells. The first `acr.run` of a session may open an interactive
   Microsoft device-code login for GCS read access; that is expected and cached.

If you have already run something that imports `ray` in this kernel, restart it
— Ray caches `RAY_AUTH_MODE` at first import.

In [ ]:
import os
import subprocess
import sys

assert sys.version_info[:2] == (3, 13), (
    f"kernel runs Python {sys.version.split()[0]}; the cluster needs 3.13.x. "
    "Select the workspace pixi environment as the kernel."
)

# Ray reads RAY_AUTH_MODE once, at its first import, and caches it. Setting it
# here works *only* because nothing has imported ray yet in a fresh kernel.
os.environ["RAY_AUTH_MODE"] = "token"
os.environ.setdefault("VAULT_ADDR", "https://vault.int.alice-bob.com")

if not os.environ.get("RAY_AUTH_TOKEN"):
    try:
        os.environ["RAY_AUTH_TOKEN"] = subprocess.run(
            ["vault", "kv", "get", "-field=token", "kv-it/prod/ray/auth-token"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError(
            "Could not mint the cluster token from Vault. In a terminal, run:\n"
            "  export VAULT_ADDR=https://vault.int.alice-bob.com\n"
            "  vault login -method=oidc role=research_core_user\n"
            "then re-run this cell (no kernel restart needed).\n"
            f"{(getattr(exc, 'stderr', '') or '').strip()}"
        ) from exc

import numpy as np

from anb_compute import ray as acr

import pathlib


def _find_repo():
    """Locate the repo checkout without hardcoding an absolute path.

    Notebooks have no __file__, so search upward from the working directory --
    which VS Code sets to either the notebook's folder or the workspace root,
    depending on `jupyter.notebookFileRoot`. Both are handled.
    """
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "src" / "floquet_lindblad").is_dir():
            return base
        for hit in sorted(base.glob("*/src/floquet_lindblad")):
            return hit.parents[1]
    raise RuntimeError(f"no checkout with src/floquet_lindblad found above {here}")


REPO = _find_repo()

# Shipped by PATH, not by name: a bare module name that resolves to a
# non-editable site-packages install ships a frozen copy, and local edits would
# never reach the worker. These prepend to the worker's sys.path, so the
# dynamiqs here shadows the image's stock 0.3.6 (which lacks `mesolve_fast`).
PKG = str(REPO / "src" / "floquet_lindblad")
# The patched dynamiqs checkout, expected beside the repo; $DYNAMIQS_SRC wins.
DYNAMIQS = os.environ.get("DYNAMIQS_SRC") or str(
    REPO.parent / "dynamiqs" / "dynamiqs"
)
for _p in (PKG, DYNAMIQS):
    if not pathlib.Path(_p).is_dir():
        raise RuntimeError(f"not a directory, cannot ship to workers: {_p}")

RUNTIME_ENV = acr.build_runtime_env(py_modules=[PKG, DYNAMIQS])

print("ready")

In [ ]:
import time

SETTINGS = dict(
    n_a=30,
    n_b=11,
    alpha_sq=4.25,
    eps_p=0.1,
    m_arnoldi_0=120,
    m_arnoldi=120,
    margin=1e-2,
)

def run_point(**kw):
    """Thin wrapper so the import happens on the worker.

    `floquet_lindblad` cannot be imported on this client: it pulls in jax and dynamiqs at
    module scope, which only the cluster image has.
    """
    from floquet_lindblad.pipeline import solve_point

    return solve_point(**kw)


t0 = time.time()
result = acr.run(
    run_point,
    num_gpus=1,
    num_cpus=8,
    runtime_env=RUNTIME_ENV,
    poll_seconds=5,
    # want_wigner: computed on the worker, since this client has no dynamiqs.
    # want_x_ritz off: at n_a*n_b = 330 it is a 330**2 complex vector we do not
    # need here, and it would only bloat the result envelope.
    want_wigner=True,
    want_x_ritz=False,
    **SETTINGS,
)
print(f"wall clock, incl. job startup: {time.time() - t0:.1f} s")
print("m_arnoldi_0 used:", result["m_arnoldi_0"], " margin used:", result["margin"])
print("rate_bf :", result["rate_bf"])
print("res     :", result["res"])

### Arnoldi convergence

In [ ]:
import matplotlib.pyplot as plt

mu_list = result["mu_list"]

fig, ax = plt.subplots(figsize=(5.5, 3.8), constrained_layout=True)
ax.scatter(range(len(mu_list)), np.abs(1 - np.real(mu_list)), s=14)
ax.set_yscale("log")
ax.set_xlabel("Arnoldi iteration")
ax.set_ylabel(r"$|1-\mathrm{Re}(\mu)|$")
ax.set_title("convergence of the bit-flip eigenvalue")
ax.grid(alpha=0.3)
plt.show()

### Ritz state in the lab frame

In [ ]:
xvec, yvec, W = result["wigner"]
alpha_sq = result["alpha_sq"]

# Same convention as dq.plot.wigner: W is indexed [x, y], clipped to +-2/pi.
vmax = 2 / np.pi
theta = np.linspace(0, 2 * np.pi, 200)
r = np.sqrt(alpha_sq)

fig, ax = plt.subplots(figsize=(5.0, 4.2), constrained_layout=True)
im = ax.pcolormesh(
    xvec, yvec, np.clip(W.T, -vmax, vmax),
    cmap="RdBu_r", vmin=-vmax, vmax=vmax, shading="auto",
)
# the |alpha| circle the cat should sit on
ax.plot(r * np.cos(theta), r * np.sin(theta), color="r", lw=1, ls="--")
ax.set_aspect("equal")
ax.set_xlabel(r"$\mathrm{Re}(\alpha)$")
ax.set_ylabel(r"$\mathrm{Im}(\alpha)$")
ax.set_title(rf"Ritz state, $|\alpha|^2 = {alpha_sq}$")
fig.colorbar(im, ax=ax, ticks=[-vmax, 0, vmax], label="W")
plt.show()